# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [1]:
# ====================
# Important Libraries
# ====================
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install -q control

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import random
import control


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [2]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [3]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1095


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [4]:
# ==================================
# Part 1: Implement a Random Policy
# ==================================

def random_policy(state, actions):
    return random.choice(actions)

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [5]:
# ===============================
# Use code to test random policy 
# ===============================
log_rows = []
for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
    print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)
    log_rows.append((curr_state, curr_action, next_reward, next_state))


# ==================================================
# Save results to TSV file named 'log-random.tsv'
# ==================================================
log_random_df = pd.DataFrame({
    'curr_state': [row[0] for row in log_rows],
    'curr_action': [row[1] for row in log_rows],
    'next_reward': [row[2] for row in log_rows],
    'next_state': [row[3] for row in log_rows],
    })

# print(log_random_df)

# Save file
log_random_df.to_csv('log-random.tsv', sep='\t', index=False)
log_random_df

CURR STATE 1149 ACTION 0 NEXT REWARD 0 NEXT STATE 1013
CURR STATE 1013 ACTION 1 NEXT REWARD 0 NEXT STATE 878
CURR STATE 878 ACTION 0 NEXT REWARD 0 NEXT STATE 870
CURR STATE 870 ACTION 0 NEXT REWARD 0 NEXT STATE 862
CURR STATE 862 ACTION 0 NEXT REWARD 0 NEXT STATE 854
CURR STATE 854 ACTION -1 NEXT REWARD 0 NEXT STATE 845
CURR STATE 845 ACTION -1 NEXT REWARD 0 NEXT STATE 708
CURR STATE 708 ACTION 1 NEXT REWARD 0 NEXT STATE 581
CURR STATE 581 ACTION 0 NEXT REWARD 0 NEXT STATE 445
CURR STATE 445 ACTION 0 NEXT REWARD 0 NEXT STATE 309
CURR STATE 309 ACTION 0 NEXT REWARD 0 NEXT STATE 173
CURR STATE 173 ACTION 0 NEXT REWARD 0 NEXT STATE 37
CURR STATE 37 ACTION 0 NEXT REWARD 0 NEXT STATE 29
CURR STATE 29 ACTION 1 NEXT REWARD 0 NEXT STATE 22
CURR STATE 22 ACTION 1 NEXT REWARD 0 NEXT STATE 15
CURR STATE 15 ACTION -1 NEXT REWARD 0 NEXT STATE 134
CURR STATE 134 ACTION -1 NEXT REWARD 0 NEXT STATE 133
CURR STATE 133 ACTION 0 NEXT REWARD 0 NEXT STATE 5
CURR STATE 5 ACTION -1 NEXT REWARD 0 NEXT STATE 4

,curr_state,curr_action,next_reward,next_state
0,1149,0,0,1013
1,1013,1,0,878
2,878,0,0,870
3,870,0,0,862
4,862,0,0,854
5,854,-1,0,845
6,845,-1,0,708
7,708,1,0,581
8,581,0,0,445
9,445,0,0,309


Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [6]:
# =================================================
# Part 2: Implement Q-Learning with Random Policy
# =================================================

alpha = 1.0
gamma = 0.9
num_states = simulator.resolution * simulator.resolution * len(simulator.DIRECTIONS)
q_values = np.zeros((num_states, 3))
q_random_rows = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        action_index = curr_action + 1
        old_value = q_values[curr_state, action_index]
        target_value = next_reward + gamma * np.max(q_values[next_state])
        new_value = old_value + alpha * (target_value - old_value)
        q_values[curr_state, action_index] = new_value
        q_random_rows.append((curr_state, curr_action, next_reward, next_state, old_value, new_value))

Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [7]:
# ================================================
# Save each step to TSV file named 'q-random.tsv'
# ================================================
q_random_df = pd.DataFrame({
    'curr_state': [row[0] for row in q_random_rows],
    'curr_action': [row[1] for row in q_random_rows],
    'next_reward': [row[2] for row in q_random_rows],
    'next_state': [row[3] for row in q_random_rows],
    'old_value': [row[4] for row in q_random_rows],
    'new_value': [row[5] for row in q_random_rows],
    })  

# print(q_random_df)

# Save file
q_random_df.to_csv("q-random.tsv", sep="\t", index=False)
q_random_df

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,1029,1,0,902,0.0,0.0
1,902,0,0,902,0.0,0.0
2,902,0,0,902,0.0,0.0
3,902,-1,0,901,0.0,0.0
4,901,0,0,773,0.0,0.0
...,...,...,...,...,...,...
32763,1088,-1,0,1088,0.0,0.0
32764,1088,0,0,1088,0.0,0.0
32765,1088,0,0,1088,0.0,0.0
32766,1088,1,0,1088,0.0,0.0


Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [8]:
# =========================================
# Part 3: Implement Epsilon Greedy Policy
# =========================================
# hard-code epsilon=0.25. this is high but the environment is deterministic.
epsilon = 0.25

def epsilon_greedy_policy(state, actions):
    if random.random() < epsilon:
        return random.choice(actions)

    # default to action 0, only replaced by a strictly better q-value
    best_action = 0
    for action in actions:
        if q_values[state, action + 1] > q_values[state, best_action + 1]:
            best_action = action
    return best_action

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [9]:
# ==============================================
# Combine epsilon greedy policy with Q-learning
# ==============================================
random.seed(6)
q_values = np.zeros((num_states, 3))
q_greedy_rows = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        action_index = curr_action + 1
        old_value = q_values[curr_state, action_index]
        target_value = next_reward + gamma * np.max(q_values[next_state])
        new_value = old_value + alpha * (target_value - old_value)
        q_values[curr_state, action_index] = new_value
        q_greedy_rows.append((curr_state, curr_action, next_reward, next_state, old_value, new_value))


# ================================================
# Save each step to TSV file named 'q-greedy.tsv'
# ================================================
q_greedy_df = pd.DataFrame({
    'curr_state': [row[0] for row in q_greedy_rows],
    'curr_action': [row[1] for row in q_greedy_rows],
    'next_reward': [row[2] for row in q_greedy_rows],
    'next_state': [row[3] for row in q_greedy_rows],
    'old_value': [row[4] for row in q_greedy_rows],
    'new_value': [row[5] for row in q_greedy_rows],
})

q_greedy_df.to_csv("q-greedy.tsv", sep="\t", index=False)
q_greedy_df

,curr_state,curr_action,next_reward,next_state,old_value,new_value
0,1026,0,0,1034,0.0,0.0
1,1034,0,0,1042,0.0,0.0
2,1042,1,0,1051,0.0,0.0
3,1051,0,0,931,0.0,0.0
4,931,0,0,811,0.0,0.0
...,...,...,...,...,...,...
32763,2040,0,0,2040,0.0,0.0
32764,2040,0,0,2040,0.0,0.0
32765,2040,0,0,2040,0.0,0.0
32766,2040,1,0,2041,0.0,0.0


Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [10]:
# ======================================
# Part 4: Extract Policy from Q-Values
# ======================================
# only include states visited in the q-greedy simulation (the curr_state column of q-greedy.tsv)
policy = {}
for state in sorted(q_greedy_df["curr_state"].unique()):
    best_action = int(np.argmax(q_values[state])) - 1
    policy[state] = best_action

# Print the extracted policy
# for state, action in policy.items():
    # print(f"State: {state}, Best Action: {action}")


# ==================================================
# Save policy to TSV file named 'policy-greedy.tsv'
# ==================================================
policy_greedy_df = pd.DataFrame({
    "state": list(policy.keys()),
    "action": list(policy.values())
    })

# print(policy_greedy_df)

# Save file
policy_greedy_df.to_csv("policy-greedy.tsv", sep="\t", index=False)
policy_greedy_df

,state,action
0,0,-1
1,1,-1
2,3,-1
3,4,-1
4,5,-1
...,...,...
1239,2041,-1
1240,2042,-1
1241,2043,-1
1242,2044,-1


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [11]:
# ================================
# Part 5: Implement Large Policy
# ================================
# q-table, trained with many rollouts
q_optimal = np.zeros((num_states, 3))
epsilon_optimal = 0.25

def optimal_training_policy(state, actions):
    if random.random() < epsilon_optimal:
        return random.choice(actions)
    best_value = max(q_optimal[state, action + 1] for action in actions)
    best_actions = [action for action in actions if q_optimal[state, action + 1] == best_value]
    return random.choice(best_actions)

for episode in range(20000):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(optimal_training_policy, max_steps=256):
        action_index = curr_action + 1
        old_value = q_optimal[curr_state, action_index]
        target_value = next_reward + gamma * np.max(q_optimal[next_state])
        q_optimal[curr_state, action_index] = old_value + alpha * (target_value - old_value)

        if next_reward > 0:
            # q-value is already updated, end rollout early
            break

# include every state
policy_optimal = {state: int(np.argmax(q_optimal[state])) - 1 for state in range(num_states)}


# ==================================================
# Save policy to TSV file named 'policy-optimal.tsv'
# ==================================================
policy_optimal_df = pd.DataFrame({
    "state": list(policy_optimal.keys()),
    "action": list(policy_optimal.values())
    })

# print(policy_optimal_df)

# Save file
policy_optimal_df.to_csv("policy-optimal.tsv", sep="\t", index=False)
policy_optimal_df

,state,action
0,0,1
1,1,0
2,2,-1
3,3,-1
4,4,-1
...,...,...
2043,2043,0
2044,2044,-1
2045,2045,-1
2046,2046,-1


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.